# 04 — Old Data Model Training

**Model:** Isolation Forest  
**Task:** historical anomaly detection

`RECOVERING` + `BROKEN` are treated as abnormal observations. This is anomaly detection, not a guaranteed early-failure predictor. The split is chronological: historical NORMAL data for training, the penultimate anomaly event for validation, and the final anomaly event for test. The threshold is tuned on validation only and frozen before test.

In [1]:
from pathlib import Path
import json,warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,classification_report,confusion_matrix,f1_score,precision_score,recall_score,roc_auc_score
warnings.filterwarnings("ignore")
BASE_DIR=Path("..")
DATA=BASE_DIR/"data"/"processed"
MODELS=BASE_DIR/"models"
INPUT=DATA/"sensor_features.csv"
MODEL_FILE=MODELS/"old_sensor_anomaly_model.joblib"
META_FILE=MODELS/"old_sensor_anomaly_metadata.json"
RANDOM_STATE=42
N_ESTIMATORS=300
EMBARGO=pd.Timedelta(minutes=120)
VAL_DAYS=3
TEST_DAYS=3

In [2]:
df=pd.read_csv(INPUT,parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
df["is_anomaly"]=(~df.machine_status.eq("NORMAL")).astype(int)
a=df.is_anomaly.eq(1)
g=(a!=a.shift(fill_value=False)).cumsum()
df["event_id"]=g.where(a)
events=(df[df.event_id.notna()].groupby("event_id",as_index=False).agg(start=("timestamp","min"),end=("timestamp","max"),rows=("timestamp","size"),recovering_count=("machine_status",lambda x:int((x=="RECOVERING").sum())),broken_count=("machine_status",lambda x:int((x=="BROKEN").sum()))))
events["event_number"]=np.arange(1,len(events)+1)
if len(events)<3: raise ValueError("Need at least 3 independent anomaly events")
print(events.to_string(index=False))

 event_id               start                 end  rows  recovering_count  broken_count  event_number
      1.0 2018-04-12 21:55:00 2018-04-13 13:39:00   945               944             1             1
      3.0 2018-04-18 00:30:00 2018-04-20 04:20:00  3111              3110             1             2
      5.0 2018-05-19 03:18:00 2018-05-20 01:10:00  1313              1312             1             3
      7.0 2018-05-25 00:30:00 2018-05-25 10:35:00   606               605             1             4
      9.0 2018-06-28 22:00:00 2018-07-04 17:50:00  8391              8390             1             5
     11.0 2018-07-08 00:11:00 2018-07-08 00:52:00    42                41             1             6
     13.0 2018-07-25 14:00:00 2018-07-25 15:15:00    76                75             1             7


In [3]:
val_event=events.iloc[-2]; test_event=events.iloc[-1]
vs,ve=val_event.start,val_event.end
ts,te=test_event.start,test_event.end
VAL_WINDOW=pd.Timedelta(days=VAL_DAYS); TEST_WINDOW=pd.Timedelta(days=TEST_DAYS)
val_start,val_end=vs-VAL_WINDOW,ve+VAL_WINDOW
test_start,test_end=ts-TEST_WINDOW,te+TEST_WINDOW
if val_end>=test_start: raise ValueError("Validation/test windows overlap; reduce VAL_DAYS or TEST_DAYS")
train_end=val_start-EMBARGO
train_mask=(df.timestamp<train_end)&df.machine_status.eq("NORMAL")
val_mask=(df.timestamp>=val_start)&(df.timestamp<=val_end)
test_mask=(df.timestamp>=test_start)&(df.timestamp<=test_end)
print("Train end:",train_end)
print("Train/Val/Test rows:",train_mask.sum(),val_mask.sum(),test_mask.sum())

Train end: 2018-07-04 22:11:00
Train/Val/Test rows: 122325 8682 8716


In [4]:
exclude={"timestamp","machine_status","event_id","is_anomaly"}
features=[c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
X=df[features].replace([np.inf,-np.inf],np.nan)
Xtr=X.loc[train_mask]; Xv=X.loc[val_mask]; Xt=X.loc[test_mask]
yv=df.loc[val_mask,"is_anomaly"].to_numpy(); yt=df.loc[test_mask,"is_anomaly"].to_numpy()
if yv.sum()==0 or yt.sum()==0: raise ValueError("Validation/test window contains no anomaly rows")
print("Feature count:",len(features))
print(Xtr.shape,Xv.shape,Xt.shape)

Feature count: 121
(122325, 121) (8682, 121) (8716, 121)


In [5]:
pipe=Pipeline([
 ("imputer",SimpleImputer(strategy="median")),
 ("scaler",StandardScaler()),
 ("model",IsolationForest(n_estimators=N_ESTIMATORS,contamination="auto",random_state=RANDOM_STATE,n_jobs=-1))
])
pipe.fit(Xtr)
score=lambda X:-pipe.decision_function(X)
strn,sv,st=score(Xtr),score(Xv),score(Xt)
print("Training complete")

Training complete


In [6]:
rows=[]
for p in np.arange(90,99.51,0.5):
    t=float(np.percentile(strn,p)); pred=(sv>=t).astype(int)
    rows.append({"percentile":p,"threshold":t,"precision":precision_score(yv,pred,zero_division=0),"recall":recall_score(yv,pred,zero_division=0),"f1":f1_score(yv,pred,zero_division=0),"alerts":int(pred.sum()),"alert_rate":float(pred.mean())})
threshold_df=pd.DataFrame(rows).sort_values(["f1","precision","recall"],ascending=False).reset_index(drop=True)
THRESHOLD=float(threshold_df.iloc[0].threshold)
print("Frozen threshold:",THRESHOLD)
display(threshold_df.head(10))

Frozen threshold: -0.04298116876616548


,percentile,threshold,precision,recall,f1,alerts,alert_rate
0,91.5,-0.042981,0.102362,0.309524,0.153846,127,0.014628
1,90.5,-0.045897,0.095238,0.333333,0.148148,147,0.016932
2,91.0,-0.044472,0.096296,0.309524,0.146893,135,0.015549
3,92.0,-0.041384,0.098361,0.285714,0.146341,122,0.014052
4,90.0,-0.047223,0.089172,0.333333,0.140704,157,0.018083
5,92.5,-0.039578,0.095652,0.261905,0.140127,115,0.013246
6,93.0,-0.037828,0.090090,0.238095,0.130719,111,0.012785
7,93.5,-0.035815,0.089109,0.214286,0.125874,101,0.011633
8,94.5,-0.031349,0.086420,0.166667,0.113821,81,0.009330
9,94.0,-0.033852,0.077778,0.166667,0.106061,90,0.010366


In [8]:
pred = (st >= THRESHOLD).astype(int)
roc = roc_auc_score(yt, st)
pr = average_precision_score(yt, st)
precision = precision_score(yt, pred, zero_division=0)
recall = recall_score(yt, pred, zero_division=0)
f1 = f1_score(yt, pred, zero_division=0)

print(f"""ROC-AUC: {roc:.4f}
PR-AUC: {pr:.4f}
Precision: {precision:.4f}
Recall: {recall:.4f}
F1: {f1:.4f}""")

print("\nConfusion Matrix:\n", confusion_matrix(yt, pred))
print("\n", classification_report(yt, pred, target_names=["NORMAL", "RECOVERING/BROKEN"], zero_division=0))

ROC-AUC: 0.9754
PR-AUC: 0.7856
Precision: 0.0701
Recall: 0.9474
F1: 0.1306

Confusion Matrix:
 [[7685  955]
 [   4   72]]

                    precision    recall  f1-score   support

           NORMAL       1.00      0.89      0.94      8640
RECOVERING/BROKEN       0.07      0.95      0.13        76

         accuracy                           0.89      8716
        macro avg       0.53      0.92      0.54      8716
     weighted avg       0.99      0.89      0.93      8716



In [9]:
teval=df.loc[test_mask,["timestamp","machine_status"]].copy()
teval["score"]=st; teval["predicted_anomaly"]=pred
target=teval[(teval.timestamp>=ts)&(teval.timestamp<=te)]
detected=bool(target.predicted_anomaly.any())
first=target.loc[target.predicted_anomaly.eq(1),"timestamp"].min() if detected else None
delay=first-ts if detected else None
print("Target event detected:",detected)
print("First detection:",first)
print("Detection delay:",delay)
print("Event alerts:",int(target.predicted_anomaly.sum()),"/",len(target))

Target event detected: True
First detection: 2018-07-25 14:00:00
Detection delay: 0 days 00:00:00
Event alerts: 72 / 76


In [10]:
MODELS.mkdir(parents=True,exist_ok=True)
joblib.dump(pipe,MODEL_FILE)
calibration_min=min(-float(np.percentile(strn,99.5)),-THRESHOLD-1e-9)
meta={"model":"IsolationForest","task":"historical_anomaly_detection","random_state":RANDOM_STATE,"n_estimators":N_ESTIMATORS,"threshold":-THRESHOLD,"threshold_score":THRESHOLD,"calibration_min":calibration_min,"threshold_selection":"validation_f1","validation_event_number":int(val_event.event_number),"test_event_number":int(test_event.event_number),"validation_event_start":str(vs),"validation_event_end":str(ve),"test_event_start":str(ts),"test_event_end":str(te),"feature_count":len(features),"feature_columns":features,"metrics":{"roc_auc":float(roc),"pr_auc":float(pr),"precision":float(precision),"recall":float(recall),"f1":float(f1),"event_detected":detected,"first_detection":str(first) if first is not None else None,"detection_delay":str(delay) if delay is not None else None}}
with META_FILE.open("w",encoding="utf-8") as f: json.dump(meta,f,indent=2)
print("Saved model:",MODEL_FILE)
print("Saved metadata:",META_FILE)

Saved model: ..\models\old_sensor_anomaly_model.joblib
Saved metadata: ..\models\old_sensor_anomaly_metadata.json


## Final audit

- The model is trained only on historical NORMAL observations.
- Missing values are imputed using statistics learned from training data only.
- The anomaly threshold is selected on validation and frozen before test; metadata stores it in the service's `decision_function` score direction.
- Test results include row-level metrics plus event detection delay.
- Because the dataset contains very few independent failures, treat this model as decision support until more real failure events are available.

**Execution order:** run all cells in `03_old_feature_engineering.ipynb`, then all cells in this notebook.